## Creating an example model (with ChatGPT) that works through fine tuning

In [1]:
import os

# Tell transformers not to use TensorFlow (we only need PyTorch here)
os.environ["TRANSFORMERS_NO_TF"] = "1"

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)

import transformers, sys, importlib

print("Python executable:", sys.executable)
print("Transformers version:", transformers.__version__)
print("TensorFlow present?", importlib.util.find_spec("tensorflow") is not None)
print("Keras present?", importlib.util.find_spec("keras") is not None)
print("tf_keras present?", importlib.util.find_spec("tf_keras") is not None)

/Users/chadadelman/anaconda3/envs/chad_env/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


Python executable: /Users/chadadelman/anaconda3/envs/chad_env/bin/python
Transformers version: 4.57.3
TensorFlow present? True
Keras present? True
tf_keras present? True


In [2]:
# Tiny toy dataset: text -> short summary

texts = [
    "Romeo and Juliet fall in love despite their families' feud.",
    "Macbeth is driven by ambition to commit terrible deeds.",
    "Hamlet struggles with indecision as he seeks revenge for his father.",
    "King Lear divides his kingdom among his daughters with tragic results.",
    "Othello is manipulated by Iago into believing Desdemona is unfaithful.",
]

summaries = [
    "Forbidden lovers in a family feud.",
    "Ambition leads Macbeth to horror.",
    "Hamlet hesitates while seeking revenge.",
    "Lear's division of his kingdom ends in tragedy.",
    "Iago tricks Othello into jealousy.",
]

raw_dataset = Dataset.from_dict({"text": texts, "summary": summaries})

# Simple train/validation split
dataset = raw_dataset.train_test_split(test_size=0.4, seed=42)
train_dataset = dataset["train"]
eval_dataset = dataset["test"]

train_dataset, eval_dataset

(Dataset({
     features: ['text', 'summary'],
     num_rows: 3
 }),
 Dataset({
     features: ['text', 'summary'],
     num_rows: 2
 }))

In [3]:
model_name = "t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("Tokenizer type:", type(tokenizer))
print("Model type:", type(model))

Tokenizer type: <class 'transformers.models.t5.tokenization_t5_fast.T5TokenizerFast'>
Model type: <class 'transformers.models.t5.modeling_t5.T5ForConditionalGeneration'>


In [4]:
max_input_length = 64
max_target_length = 32

def preprocess_function(batch):
    # T5 likes a task prefix, e.g. "summarize: "
    inputs = ["summarize: " + t for t in batch["text"]]
    model_inputs = tokenizer(
        inputs,
        max_length=max_input_length,
        truncation=True,
        padding="max_length",
    )

    labels = tokenizer(
        batch["summary"],
        max_length=max_target_length,
        truncation=True,
        padding="max_length",
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_train = train_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=["text", "summary"],
)

tokenized_eval = eval_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=["text", "summary"],
)

tokenized_train[:2]


Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

{'input_ids': [[21603,
   10,
   2671,
   312,
   291,
   14514,
   7,
   112,
   14740,
   859,
   112,
   16649,
   28,
   17414,
   772,
   5,
   1,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0],
  [21603,
   10,
   2143,
   346,
   189,
   19,
   6737,
   57,
   12517,
   12,
   10042,
   9412,
   20,
   15,
   26,
   7,
   5,
   1,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0]],
 'attention_mask': [[1,
   1,
   1,
   1,
   1,
   1,
   1,
   1,
   1,
   1,
   1,
   1,
   1,
   1,
   1,
   1,
   1,
   0,
   0,
   0,
   0,
   

In [5]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
)

print("Data collator ready:", type(data_collator))

Data collator ready: <class 'transformers.data.data_collator.DataCollatorForSeq2Seq'>


In [6]:
# 8. Manual training loop (no Trainer, no Hugging Face Hub)

import torch
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# DataLoader for training
train_dataloader = DataLoader(
    tokenized_train,
    batch_size=2,
    shuffle=True,
    collate_fn=data_collator,
)

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-7)

num_epochs = 8

model.train()
for epoch in range(num_epochs):
    total_loss = 0.0
    for step, batch in enumerate(train_dataloader):
        # Move batch to device
        batch = {k: v.to(device) for k, v in batch.items()}

        outputs = model(**batch)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        if (step + 1) % 5 == 0:
            print(f"Epoch {epoch+1}, Step {step+1}, Loss: {loss.item():.4f}")

    avg_loss = total_loss / len(train_dataloader)
    print(f"Epoch {epoch+1} finished. Average loss: {avg_loss:.4f}")

print("Training complete (manual loop).")

Epoch 1 finished. Average loss: 12.2187
Epoch 2 finished. Average loss: 11.3310
Epoch 3 finished. Average loss: 12.7534
Epoch 4 finished. Average loss: 11.4021
Epoch 5 finished. Average loss: 11.7190
Epoch 6 finished. Average loss: 12.0548
Epoch 7 finished. Average loss: 12.1660
Epoch 8 finished. Average loss: 11.5787
Training complete (manual loop).


In [7]:
# 9. Quick test generation after manual fine-tuning

model.eval()

test_text = "A prince wrestles with doubt and morality as he considers avenging his father's murder."

inputs = tokenizer(
    "summarize: " + test_text,
    return_tensors="pt",
    truncation=True,
    padding=True,
).to(device)

with torch.no_grad():
    generated_ids = model.generate(
        **inputs,
        max_length=32,
        num_beams=4,
    )

generated_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
print("INPUT:", test_text)
print("SUMMARY:", generated_text)

INPUT: A prince wrestles with doubt and morality as he considers avenging his father's murder.
SUMMARY: prince considers avenging his father's murder. he considers avenging his father's murder.
